# Clinical NLP Training on Kaggle

Notebook này train XLM-R NER bằng annotation thật, reload checkpoint vừa train,
chạy entity linking/assertion và tạo `/kaggle/working/output.zip`.

Trước khi Run All: attach Dataset có input + annotation, bật GPU, và bật
Internet nếu không attach sẵn code/model. Xem `KAGGLE_RUNBOOK.md`.

## 1. Runtime and repository bootstrap

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys

# [CRITICAL KAGGLE PATCH] Prevent TensorFlow and JAX from pre-allocating 100% of GPU memory
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['JAX_PLATFORM_NAME'] = 'cpu'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'


IS_KAGGLE = Path("/kaggle/input").is_dir()
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

GITHUB_REPO_URL = "https://github.com/takumi612/AI-Race-Viettel.git"
GITHUB_BRANCH = "main"
PROJECT_ROOT_OVERRIDE = ""
MODEL_NAME_OR_PATH_OVERRIDE = ""
INPUT_SOURCE_OVERRIDE = ""
TRAIN_SOURCE_OVERRIDE = ""
INSTALL_MISSING_DEPENDENCIES = True
FAST_DEV_RUN = False
REQUIRE_TRAINING_DATA = True
REQUIRE_GPU = True

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")

from datetime import datetime, timezone
import traceback

CURRENT_STEP = None
STEP_START = 'STEP_START'
STEP_END = 'STEP_END'
STEP_ERROR = 'STEP_ERROR'
def log_step(step: str, event: str, message: str = "", **details):
    payload = {"time_utc": datetime.now(timezone.utc).isoformat(), "step": step, "event": event, "message": message, **details}
    marker = {'START': STEP_START, 'END': STEP_END, 'ERROR': STEP_ERROR}.get(event, f'STEP_{event}')
    print(f"[{marker}] {json.dumps(payload, ensure_ascii=False, default=str)}", flush=True)

def log_error(step: str, exc: BaseException):
    log_step(step, "ERROR", str(exc), exception_type=type(exc).__name__, traceback=traceback.format_exc())

CURRENT_STEP = '1'
log_step(CURRENT_STEP, 'START', 'Runtime/bootstrap started')

def _is_project(path: Path) -> bool:
    return (path / "clinical_nlp_lab").is_dir() and (path / "artifacts/config.json").is_file()

project_candidates = []
if PROJECT_ROOT_OVERRIDE.strip():
    project_candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())
project_candidates.append(Path.cwd() / "v2")
project_candidates.append(Path.cwd())
if IS_KAGGLE:
    for marker in KAGGLE_INPUT_ROOT.rglob("clinical_nlp_lab"):
        if marker.is_dir():
            project_candidates.extend([marker.parent, marker.parent.parent / "v2"])

PROJECT_ROOT = next((path.resolve() for path in project_candidates if _is_project(path)), None)
clone_dir = KAGGLE_WORKING_ROOT / "AI-Race-Viettel"
if PROJECT_ROOT is None and IS_KAGGLE:
    clone_candidates = [clone_dir / "v2", clone_dir]
    if clone_dir.exists() and not any(_is_project(path) for path in clone_candidates):
        raise RuntimeError(f"Clone destination exists but is not a valid project: {clone_dir}")
    if not clone_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(clone_dir)],
            check=True,
        )
    PROJECT_ROOT = next((path.resolve() for path in clone_candidates if _is_project(path)), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project code not found. Enable Internet for Git clone or attach a code Dataset and set PROJECT_ROOT_OVERRIDE."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    CODE_REVISION = subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    CODE_REVISION = 'attached-code-dataset'

required_imports = {
    "torch": "torch",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bm25s": "bm25s",
    "faiss-cpu": "faiss",
    "sentence-transformers": "sentence_transformers"
}
missing = [package for package, module in required_imports.items() if importlib.util.find_spec(module) is None]
if IS_KAGGLE and missing and INSTALL_MISSING_DEPENDENCIES:
    requirements = PROJECT_ROOT / "requirements-kaggle.txt"
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
        importlib.invalidate_caches()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f"Could not install {missing}. Enable Internet or attach an environment/model Dataset."
        ) from exc

missing_after = [package for package, module in required_imports.items() if importlib.util.find_spec(module) is None]
DEPENDENCIES_READY = not missing_after
if IS_KAGGLE and missing_after:
    raise RuntimeError(f"Missing training dependencies after setup: {missing_after}")

# vLLM is an optional inference runtime, intentionally separate from core requirements.
VLLM_PACKAGE_SPEC = 'vllm==0.25.1'
VLLM_INSTALL_ERROR = None
if IS_KAGGLE and importlib.util.find_spec('vllm') is None and INSTALL_MISSING_DEPENDENCIES:
    try:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', VLLM_PACKAGE_SPEC],
            check=True,
        )
        importlib.invalidate_caches()
    except subprocess.CalledProcessError as exc:
        VLLM_INSTALL_ERROR = str(exc)
VLLM_READY = importlib.util.find_spec('vllm') is not None
# AutoAWQ is not required: vLLM loads AWQ checkpoints directly.

print()
print("="*55)
print("📌 STEP 1: RUNTIME AND ENVIRONMENT BOOTSTRAP LOG")
print("="*55)
print(f"Is Kaggle Environment : {IS_KAGGLE}")
print(f"Project Root Directory: {PROJECT_ROOT}")
print(f"Code Revision         : {CODE_REVISION}")
print(f"Python Version       : {sys.version.split()[0]}")
try:
    import torch
    print(f"PyTorch Version      : {torch.__version__}")
    print(f"CUDA Available       : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
        print(f"GPU Total VRAM       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
except Exception:
    pass
print(f"Dependencies Ready   : {DEPENDENCIES_READY}")
print(f"vLLM Inference Ready : {VLLM_READY}")
if VLLM_INSTALL_ERROR:
    print(f"vLLM Install Error    : {VLLM_INSTALL_ERROR}")
if missing_after:
    print(f"Missing Dependencies : {missing_after}")
print("="*55)
log_step('1', 'END', 'Runtime/bootstrap completed', project_root=str(PROJECT_ROOT), dependencies_ready=DEPENDENCIES_READY, vllm_ready=VLLM_READY)
print()


## 2. Discover attached input and annotations

In [ ]:
CURRENT_STEP = '2'
log_step(CURRENT_STEP, 'START', 'Discovering input and annotation sources')
def _has_text_files(path: Path) -> bool:
    return path.is_dir() and any(path.glob("*.txt"))

def _is_archive_path(path: Path, search_root: Path) -> bool:
    try:
        relative_parts = path.relative_to(search_root).parts
    except ValueError:
        relative_parts = path.parts
    return bool({"test", "archive", "runtime_evidence"} & {part.lower() for part in relative_parts})

def _has_training_layout(path: Path) -> bool:
    if not path.is_dir():
        return False
    text_stems = {item.stem for item in path.glob("*.txt")}
    json_stems = {item.stem for item in path.glob("*.json")}
    direct_pairs = bool(text_stems & json_stems)
    split_pairs = _has_text_files(path / "input") and (path / "gt").is_dir() and any((path / "gt").glob("*.json"))
    return direct_pairs or split_pairs

def _candidate_priority(path: Path) -> tuple[int, str]:
    normalized = str(path).lower()
    return (0 if 'ai-race-clinical-data' in normalized else 1, normalized)

search_roots = [PROJECT_ROOT]
if IS_KAGGLE:
    search_roots = [path for path in KAGGLE_INPUT_ROOT.iterdir() if path.is_dir()] + search_roots
else:
    search_roots.extend(ancestor for ancestor in PROJECT_ROOT.parents if (ancestor / "input.zip").is_file())

if INPUT_SOURCE_OVERRIDE.strip():
    input_candidates = [Path(INPUT_SOURCE_OVERRIDE).expanduser()]
else:
    input_candidates = []
    for root in search_roots:
        input_candidates.extend(path for path in root.rglob("input.zip") if not _is_archive_path(path, root))
        input_candidates.extend(
            path for path in root.rglob("input")
            if _has_text_files(path)
            and path.parent.name.lower() not in {"synthetic_train_v1", "train"}
            and not _is_archive_path(path, root)
        )
    input_candidates = sorted(set(input_candidates), key=_candidate_priority)
INPUT_SOURCE = next((path.resolve() for path in input_candidates if path.is_file() or _has_text_files(path)), None)
if INPUT_SOURCE is None:
    raise FileNotFoundError(
        "Real inference input not found. Attach a Dataset containing input.zip or input/<id>.txt."
    )

if TRAIN_SOURCE_OVERRIDE.strip():
    train_candidates = [Path(TRAIN_SOURCE_OVERRIDE).expanduser()]
else:
    train_candidates = []
    for root in search_roots:
        train_candidates.extend(path for path in root.rglob("train") if not _is_archive_path(path, root))
        train_candidates.extend(path for path in root.rglob("synthetic_train_v1") if not _is_archive_path(path, root))
    train_candidates = sorted(set(train_candidates), key=_candidate_priority)

# Local-only fixture lets the generated notebook be executed as a smoke test.
if not IS_KAGGLE and not any(_has_training_layout(path) for path in train_candidates):
    local_fixture_candidates = [
        ancestor / "test/archive/tests/fixtures/paired_annotations"
        for ancestor in (PROJECT_ROOT, *PROJECT_ROOT.parents)
    ]
    local_fixture = next(
        (path for path in local_fixture_candidates if _has_training_layout(path)),
        None,
    )
    if local_fixture is not None:
        train_candidates.insert(0, local_fixture)

TRAIN_SOURCE = next((path.resolve() for path in train_candidates if _has_training_layout(path)), None)
if TRAIN_SOURCE is None and REQUIRE_TRAINING_DATA:
    raise FileNotFoundError(
        "No annotated training data found. Attach train/*.txt + *.json or synthetic_train_v1/input + gt."
    )

if IS_KAGGLE:
    RUN_ROOT = KAGGLE_WORKING_ROOT
else:
    RUN_ROOT = PROJECT_ROOT / "test/runtime_evidence/kaggle_notebook_simulation"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
TRAINING_ROOT = RUN_ROOT / "training_artifacts"
NER_MODEL_DIR = TRAINING_ROOT / "ner_model"
OUTPUT_DIR = RUN_ROOT / "output"
DIAGNOSTICS_DIR = RUN_ROOT / "diagnostics"
OUTPUT_ZIP = RUN_ROOT / "output.zip"
TRAINED_ARTIFACTS_ZIP = RUN_ROOT / "trained_ner_artifacts.zip"

print()
print("="*55)
print("📂 STEP 2: DATASET DISCOVERY LOG")
print("="*55)
print(f"Inference Input Source: {INPUT_SOURCE}")
print(f"Training Data Source  : {TRAIN_SOURCE}")
print(f"Run Working Directory : {RUN_ROOT}")
print(f"Output Zip Target     : {OUTPUT_ZIP}")
print("="*55)
log_step(CURRENT_STEP, 'END', 'Dataset discovery completed', input_source=str(INPUT_SOURCE), train_source=str(TRAIN_SOURCE))
print()

## 3. Validate data and check GPU

In [ ]:
from clinical_nlp_lab.config import load_config, set_reproducible_seed
from clinical_nlp_lab.data import (
    load_annotated_documents,
    load_input_documents,
    validate_documents,
)
from clinical_nlp_lab.schema import write_json

CURRENT_STEP = '3'
log_step(CURRENT_STEP, 'START', 'Validating documents and checking GPU')
CONFIG = load_config(PROJECT_ROOT / "artifacts/config.json")
SEED_STATUS = set_reproducible_seed(int(CONFIG["seed"]))
INPUT_DOCUMENTS = load_input_documents(INPUT_SOURCE)
ANNOTATED_DOCUMENTS = load_annotated_documents(TRAIN_SOURCE) if TRAIN_SOURCE else []
ANNOTATION_REPORT = validate_documents(ANNOTATED_DOCUMENTS)
if not ANNOTATION_REPORT["is_valid"]:
    raise ValueError(f"Training annotation validation failed: {ANNOTATION_REPORT['errors'][:10]}")
if REQUIRE_TRAINING_DATA and not ANNOTATED_DOCUMENTS:
    raise ValueError("Training is required but no annotated documents were loaded")

VALIDATION_DOCUMENT_COUNT = max(1, round(len(ANNOTATED_DOCUMENTS) * float(CONFIG["validation_fraction"]))) if len(ANNOTATED_DOCUMENTS) > 1 else 0
TRAIN_DOCUMENT_COUNT = len(ANNOTATED_DOCUMENTS) - VALIDATION_DOCUMENT_COUNT
if FAST_DEV_RUN:
    TRAIN_DOCUMENT_COUNT = min(16, TRAIN_DOCUMENT_COUNT)
    VALIDATION_DOCUMENT_COUNT = min(4, VALIDATION_DOCUMENT_COUNT)

GPU_STATUS = {"available": False, "name": None}
if DEPENDENCIES_READY:
    import torch
    GPU_STATUS = {
        "available": torch.cuda.is_available(),
        "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
if IS_KAGGLE and REQUIRE_GPU and not GPU_STATUS["available"]:
    raise RuntimeError("GPU is required. Open Kaggle Settings and select a GPU accelerator.")

print()
print("="*55)
print("📊 STEP 3: DATASET STATISTICS & VALIDATION LOG")
print("="*55)
print(f"Input Inference Docs  : {len(INPUT_DOCUMENTS)}")
print(f"Annotated Train Docs  : {len(ANNOTATED_DOCUMENTS)}")
print(f"Train Split Docs      : {TRAIN_DOCUMENT_COUNT}")
print(f"Validation Split Docs : {VALIDATION_DOCUMENT_COUNT}")
print(f"Total GT Entities     : {ANNOTATION_REPORT.get('entity_count', 0)}")
if ANNOTATION_REPORT.get("type_counts"):
    print("Entity Type Breakdown :")
    for etype, count in ANNOTATION_REPORT["type_counts"].items():
        print(f"  - {etype:<15}: {count}")
print(f"GPU Accelerator Status: {GPU_STATUS}")
print(f"Reproducible Seed     : {CONFIG['seed']} {SEED_STATUS}")
log_step(CURRENT_STEP, 'END', 'Validation completed', input_documents=len(INPUT_DOCUMENTS), annotated_documents=len(ANNOTATED_DOCUMENTS), gpu=GPU_STATUS)
print("="*55)
print()

## 4. Training configuration

In [ ]:
CURRENT_STEP = '4'
log_step(CURRENT_STEP, 'START', 'Preparing NER training configuration')
def _looks_like_xlmr_model(path: Path) -> bool:
    config_path = path / "config.json"
    if not config_path.is_file():
        return False
    try:
        payload = json.loads(config_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return payload.get("model_type") in {"xlm-roberta", "roberta"} and (
        (path / "tokenizer.json").is_file()
        or (path / "sentencepiece.bpe.model").is_file()
    )

attached_models = []
if IS_KAGGLE:
    attached_models = sorted(
        {path.parent for path in KAGGLE_INPUT_ROOT.rglob("config.json") if _looks_like_xlmr_model(path.parent)},
        key=_candidate_priority,
    )

if MODEL_NAME_OR_PATH_OVERRIDE.strip():
    MODEL_SOURCE = MODEL_NAME_OR_PATH_OVERRIDE
elif attached_models:
    MODEL_SOURCE = str(attached_models[0])
else:
    MODEL_SOURCE = str(CONFIG["ner_model_name"])

NER_EPOCHS = 1 if FAST_DEV_RUN else int(CONFIG["ner_epochs"])
TRAIN_BATCH_SIZE = 2 if FAST_DEV_RUN else int(CONFIG["batch_size"])
LEARNING_RATE = float(CONFIG["learning_rate"])

print()
print("="*55)
print("⚙️ STEP 4: TRAINING HYPERPARAMETERS & CONFIGURATION")
print("="*55)
print(f"Base Model Source     : {MODEL_SOURCE}")
print(f"Training Epochs       : {NER_EPOCHS}")
print(f"Batch Size (per GPU)  : {TRAIN_BATCH_SIZE}")
print(f"Learning Rate         : {LEARNING_RATE}")
print(f"Max Sequence Length   : {CONFIG.get('max_length')}")
print(f"Window Stride         : {CONFIG.get('stride')}")
print(f"Fast Dev Mode Run     : {FAST_DEV_RUN}")
log_step(CURRENT_STEP, 'END', 'Training configuration ready', model_source=str(MODEL_SOURCE), epochs=NER_EPOCHS, batch_size=TRAIN_BATCH_SIZE, learning_rate=LEARNING_RATE)
print("="*55)
print()

## 4.5 Check GPU Memory (Diagnostic)

Chạy cell này để đảm bảo VRAM hoàn toàn trống trước khi train.
Nếu thấy `Memory-Usage` lớn hơn 1GB dù chưa train, bạn đang bị kẹt tiến trình ẩn (Zombie Process). Hãy bấm nút **Power Off** ở góc phải Kaggle để tắt hẳn máy ảo và bật lại.

In [ ]:
CURRENT_STEP = '4.5'
log_step(CURRENT_STEP, 'START', 'Collecting GPU memory diagnostic')
import subprocess
import sys
print("="*55)
print("🔍 GPU MEMORY DIAGNOSTIC")
print("="*55)
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except Exception as e:
    print(f"Không thể chạy nvidia-smi: {e}")
log_step(CURRENT_STEP, 'END', 'GPU diagnostic completed')
print("="*55)

## 5. Train XLM-R token classifier

In [ ]:
import subprocess
import sys
import json
from pathlib import Path
from clinical_nlp_lab.schema import write_json

CURRENT_STEP = '5'
log_step(CURRENT_STEP, 'START', 'Starting NER subprocess training', train_source=str(TRAIN_SOURCE), output_dir=str(TRAINING_ROOT), epochs=NER_EPOCHS, model_source=str(MODEL_SOURCE))

if DEPENDENCIES_READY and ANNOTATED_DOCUMENTS:
    if (NER_MODEL_DIR / "model.safetensors").exists() or (NER_MODEL_DIR / "pytorch_model.bin").exists():
        print(f"[SKIP] Found existing NER weights at {NER_MODEL_DIR}, skipping training.")
        NER_TRAINING_RESULT = {"trained": False, "reason": "checkpoint_exists", "model_dir": str(NER_MODEL_DIR)}
    else:
        print("Bắt đầu Training qua Subprocess để chống phân mảnh VRAM...")
        script_path = PROJECT_ROOT / "scripts" / "train_ner_subprocess.py"
        config_path = PROJECT_ROOT / "artifacts" / "config.json"
        
        cmd = [
            sys.executable, str(script_path),
            "--train-source", str(TRAIN_SOURCE),
            "--output-dir", str(TRAINING_ROOT),
            "--config-path", str(config_path),
            "--model-source", str(MODEL_SOURCE),
            "--fast-dev-run", str(FAST_DEV_RUN)
        ]
        
        try:
            subprocess.run(cmd, check=True)
            print("Subprocess Training hoàn tất và đã thoát. VRAM trống hoàn toàn.")
            # Đọc lại file kết quả từ subprocess
            NER_TRAINING_RESULT = json.loads((TRAINING_ROOT / "training_result.json").read_text(encoding='utf-8'))
        except subprocess.CalledProcessError as e:
            log_error(CURRENT_STEP, e)
            raise RuntimeError("Training Subprocess thất bại!") from e
else:
    NER_TRAINING_RESULT = {"trained": False, "reason": "No training documents"}

TRAINING_ROOT.mkdir(parents=True, exist_ok=True)
write_json(TRAINING_ROOT / "training_result.json", NER_TRAINING_RESULT)
log_step(CURRENT_STEP, 'END', 'NER subprocess training completed', trained=NER_TRAINING_RESULT.get('trained'), reason=NER_TRAINING_RESULT.get('reason'), best_metric=NER_TRAINING_RESULT.get('best_metric'))

print('='*55)
print('🚀 STEP 5: XLM-R NER MODEL TRAINING EXECUTION')
print('='*55)
print(f'✅ Training Status     : {NER_TRAINING_RESULT.get("trained")}')


## 6. Package the trained checkpoint

In [ ]:
CURRENT_STEP = '6'
log_step(CURRENT_STEP, 'START', 'STEP 6: Packaging trained NER checkpoint', training_root=str(TRAINING_ROOT), archive=str(TRAINED_ARTIFACTS_ZIP))
archive_base = TRAINED_ARTIFACTS_ZIP.with_suffix("")
created_archive = shutil.make_archive(str(archive_base), "zip", root_dir=TRAINING_ROOT)
assert Path(created_archive).is_file()
zip_size_mb = Path(created_archive).stat().st_size / (1024 * 1024)
log_step(CURRENT_STEP, 'END', 'NER checkpoint packaged', archive=str(created_archive), size_mb=round(zip_size_mb, 2))
print(f"📦 Checkpoint Packaged: {created_archive} ({zip_size_mb:.2f} MB)")

## 6.5 Garbage collection

In [ ]:
import gc
import torch
CURRENT_STEP = '6.5'
log_step(CURRENT_STEP, 'START', 'Releasing training memory before inference')
gc.collect()
log_step(CURRENT_STEP, 'END', 'Python garbage collection completed')
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 GPU Memory Cleaned (empty_cache & gc.collect).")

## 7. Inference with the newly trained NER model

In [ ]:
CURRENT_STEP = '7'
log_step(CURRENT_STEP, 'START', 'Starting inference pipeline', input_source=str(INPUT_SOURCE), output_dir=str(OUTPUT_DIR), active_ner=str(NER_MODEL_DIR))
# vLLM was checked separately during bootstrap; AutoAWQ is not required.
print(f'vLLM ready before inference: {VLLM_READY}')

from clinical_nlp_lab.pipeline import run_inference

ACTIVE_NER_MODEL = NER_MODEL_DIR if (NER_TRAINING_RESULT.get("trained") or NER_TRAINING_RESULT.get("reason") == "checkpoint_exists") else None
try:
    INFERENCE_SUMMARY = run_inference(
    INPUT_SOURCE,
    OUTPUT_DIR,
    PROJECT_ROOT / "artifacts",
    create_zip=True,
    diagnostics_dir=DIAGNOSTICS_DIR,
    zip_path=OUTPUT_ZIP,
    ner_model_dir=ACTIVE_NER_MODEL,
        train_documents=ANNOTATED_DOCUMENTS,
    )
except Exception as exc:
    log_error(CURRENT_STEP, exc)
    raise
log_step(CURRENT_STEP, 'END', 'Inference pipeline completed', documents=INFERENCE_SUMMARY.get('document_count'), entities=INFERENCE_SUMMARY.get('submission_entity_count'), llm_reranker=INFERENCE_SUMMARY.get('llm_reranker_enabled'), llm_assertion=INFERENCE_SUMMARY.get('llm_assertion_enabled'))

print()
print("="*55)
print("⚡ STEP 7: INFERENCE PIPELINE EXECUTION LOG")
print("="*55)
print(f"Active NER Model      : {INFERENCE_SUMMARY.get('active_ner')}")
print(f"Documents Inferred    : {INFERENCE_SUMMARY.get('document_count')}")
print(f"Total Entities Output : {INFERENCE_SUMMARY.get('submission_entity_count')}")
print(f"LLM Reranker Enabled : {INFERENCE_SUMMARY.get('llm_reranker_enabled')}")
print(f"LLM Assertion Enabled: {INFERENCE_SUMMARY.get('llm_assertion_enabled')}")
if INFERENCE_SUMMARY.get('llm_fallback_reason'):
    print(f"LLM Fallback Reason  : {INFERENCE_SUMMARY['llm_fallback_reason']}")
if INFERENCE_SUMMARY.get("internal_type_counts"):
    print("Entity Type Breakdown :")
    for etype, count in INFERENCE_SUMMARY["internal_type_counts"].items():
        print(f"  - {etype:<15}: {count}")
print(f"Output Zip Path       : {INFERENCE_SUMMARY.get('zip_path')}")
print("="*55)
print()

## 8. Validate submission and write run manifest

In [ ]:
import zipfile
from clinical_nlp_lab.schema import validate_submission_payload

CURRENT_STEP = '8'
log_step(CURRENT_STEP, 'START', 'STEP 8: Validating submission archive and writing manifest', output_zip=str(OUTPUT_ZIP))
schema_errors = {}
for document in INPUT_DOCUMENTS:
    prediction_path = OUTPUT_DIR / f"{document.document_id}.json"
    prediction = json.loads(prediction_path.read_text(encoding="utf-8"))
    errors = validate_submission_payload(prediction, document.raw_text)
    if errors:
        schema_errors[document.document_id] = errors
if schema_errors:
    validation_error = ValueError(f"Submission schema errors: {list(schema_errors.items())[:3]}")
    log_error(CURRENT_STEP, validation_error)
    raise validation_error

with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    zip_names = archive.namelist()
    assert len(zip_names) == len(INPUT_DOCUMENTS)
    assert all(name.startswith("output/") and not name.startswith("output/output/") for name in zip_names)
    assert archive.testzip() is None

RUN_MANIFEST = {
    "trained": bool(NER_TRAINING_RESULT.get("trained")),
    "training_best_metric": NER_TRAINING_RESULT.get("best_metric"),
    "active_ner": INFERENCE_SUMMARY.get("active_ner"),
    "train_documents": TRAIN_DOCUMENT_COUNT,
    "validation_documents": VALIDATION_DOCUMENT_COUNT,
    "input_documents": len(INPUT_DOCUMENTS),
    "submission_entities": INFERENCE_SUMMARY["submission_entity_count"],
    "schema_error_count": 0,
    "offset_error_count": INFERENCE_SUMMARY["offset_error_count"],
    "llm_reranker_enabled": INFERENCE_SUMMARY.get("llm_reranker_enabled"),
    "llm_assertion_enabled": INFERENCE_SUMMARY.get("llm_assertion_enabled"),
    "llm_fallback_reason": INFERENCE_SUMMARY.get("llm_fallback_reason"),
    "code_revision": CODE_REVISION,
    "output_zip": str(OUTPUT_ZIP),
    "trained_artifacts_zip": str(TRAINED_ARTIFACTS_ZIP),
}
write_json(RUN_ROOT / "run_manifest.json", RUN_MANIFEST)
log_step(CURRENT_STEP, 'END', 'Submission validation and manifest completed', zip_entries=len(zip_names), schema_errors=len(schema_errors), manifest=str(RUN_ROOT / 'run_manifest.json'))
print(RUN_MANIFEST)

## 9. Download results (STEP 9)

Kaggle outputs:

- `/kaggle/working/output.zip`
- `/kaggle/working/trained_ner_artifacts.zip` — gói checkpoint/weights NER vừa train (`ner_model/model.safetensors` hoặc `pytorch_model.bin`, `config.json`, tokenizer và `training_result.json`).
- `/kaggle/working/training_artifacts/ner_model/` — thư mục checkpoint NER gốc trong phiên chạy.

Lưu ý: Qwen/vLLM reranker chỉ được tải để inference, không phải mô hình được train trong notebook nên không có weight mới để đóng gói.
- `/kaggle/working/run_manifest.json`
- `/kaggle/working/diagnostics/run_summary.json` — trạng thái NER/retrieval/LLM và lý do fallback nếu có.

Chọn **Save Version → Save & Run All**, sau đó tải file từ tab Output.